In [ ]:
!pip install -q torch transformers datasets rank_bm25 openai matplotlib pillow

In [ ]:
import ast
import base64
import io
import json
import os
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from openai import OpenAI
from PIL import Image
from rank_bm25 import BM25Okapi
from transformers import ColQwen2ForRetrieval, ColQwen2Processor

warnings.filterwarnings("ignore")
os.environ["OPENAI_API_KEY"] = ""

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "vidore/colqwen2-v1.0-hf"
DATASET_NAME = "vidore/infovqa_test_subsampled"
NUM_PAGES = 100
SEED = 0
torch.manual_seed(SEED)
print(f"DEVICE = {DEVICE}")

client = OpenAI()
MODEL = "gpt-5-mini"

# Visual Document Retrieval (ColPali)

Retrieval over PDFs, slides and infographics usually starts by turning pages into text. This
notebook skips that step. A vision-language model embeds each **page image** as a bag of patch
vectors, and queries are scored against those patches by late interaction. It is built and
measured on real infographics with ColQwen2, against an OCR + BM25 pipeline and a single-vector
baseline.

---

# 1) Introduction

## 1.1 The problem

Given a question ("Which category of apps is least popular?") and a corpus of document pages, rank
the page that answers it first. The answer often lives in a chart, a table cell or an icon caption.

## 1.2 Where the text pipeline fails

| Stage of OCR-based RAG | What gets lost |
|---|---|
| OCR | Charts, icons, colour coding; reading order on multi-column layouts |
| Layout detection + chunking | The link between a number and the axis or legend that gives it meaning |
| Text embedding of chunks | Anything the chunker split apart; visual emphasis |
| Pipeline itself | Three models to run and maintain; minutes per page for high-quality OCR |

## 1.3 The core idea

Feed the page image to a VLM, keep the last hidden state of **every visual token**, project it to
128 dimensions, and store all of them. A query is embedded the same way, token by token. The score
is ColBERT's **MaxSim**: every query token finds its best-matching patch.

```
page image ─► Qwen2-VL vision tower ─► ~700 visual tokens ─┐
                                                          ├─► Qwen2 LM ─► last hidden states ─► Linear(1536→128) ─► L2 norm
prompt "Describe the image." ──────────────────────────────┘
                                                      page = (num_tokens, 128)  ← stored
query "Query: … <pad>×10" ─► Qwen2 LM ─► … ─► (num_query_tokens, 128)
score = Σ_query tokens  max_page tokens  q·d
```

## 1.4 Findings from the past year

| Model / work | Date | What is new |
|---|---|---|
| **ColPali** | ICLR 2025 | PaliGemma-3B + linear projection of patch outputs to multi-vectors; ViDoRe benchmark |
| **ColQwen2 / ColQwen2.5** | 2024–25 | Qwen2-VL-2B backbone, dynamic resolution without aspect-ratio distortion; ~87–89 ViDoRe (third-party figures) |
| **ColSmol** | 2025 | SmolVLM backbone, fastest of the family |
| **jina-embeddings-v4** | 2025 | Single- and multi-vector outputs from one VLM backbone |
| `colpali-engine` 0.3.17 | Jun 2026 | ColVision family plus bi-encoder retrievers in one package |
| Structural Anchor Pruning | Jan 2026 | Training-free compression of multi-vector page indexes (evaluated on ColPali, ColQwen2, Jina v4) |

## 1.5 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | 100 InfoVQA infographics + their questions (ViDoRe), ColQwen2 (2.2B) |
| 3) The math | MaxSim, contrastive training loss, nDCG, index size |
| 4) From scratch | Page embeddings from the VLM's hidden states; batched MaxSim; nDCG@5; OCR text + BM25 |
| 5) End-to-end | Three retrievers on the same queries; where a query token looks; answer generation over the top page |
| 6) Variants | Patch pooling and binarisation: index size vs accuracy |
| 7) Practical notes | Vector stores, candidate generation, cost |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Data: InfoVQA from the ViDoRe benchmark

Each row is a question about one infographic, with the page image and the **AWS Textract** OCR
output of that page. The first 100 rows give 100 pages and 100 queries, and each query has exactly
one relevant page. Infographics are tall, dense and visual, which is the hardest case for OCR
pipelines.

In [ ]:
dataset = load_dataset(DATASET_NAME, split="test").select(range(NUM_PAGES))
pages = [row["image"].convert("RGB") for row in dataset]
queries = dataset["query"]
sizes = np.array([page.size for page in pages])
print(f"{len(pages)} pages, {len(queries)} queries")
print(
    f"page size (W x H): median {np.median(sizes, axis=0).astype(int)}, "
    f"tallest {sizes[:, 1].max()} px"
)
print(f"example query: {queries[0]!r}")

fig, axes = plt.subplots(1, 5, figsize=(16, 7))
for ax, page, query in zip(axes, pages, queries):
    ax.imshow(page)
    ax.set_title(query[:40] + "…", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2.2 Model

`vidore/colqwen2-v1.0-hf` is Qwen2-VL-2B with a LoRA fine-tune and a 1536 → 128 projection,
trained contrastively on (query, page) pairs. Qwen2-VL keeps each page's aspect ratio. It
splits the image into 14 px patches and merges 2 × 2 of them into one token, so a page costs up
to `longest_edge / 28²` ≈ 768 tokens.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


start = time.time()
model = (
    ColQwen2ForRetrieval.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
    .to(DEVICE)
    .eval()
)
processor = ColQwen2Processor.from_pretrained(MODEL_NAME)
print(
    f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f} B params | projection "
    f"{tuple(model.embedding_proj_layer.weight.shape)} "
    f"| load {time.time() - start:.0f} s | "
    f"device memory {device_memory_gb():.2f} GB"
)

---

# 3) The math

## 3.1 Late interaction (MaxSim)

A query with token embeddings $q_1, \dots, q_n$ and a page with token embeddings $d_1, \dots, d_m$
(all L2-normalised, $\in \mathbb{R}^{128}$) score

$$
s(q, d) = \sum_{i=1}^{n} \max_{j \le m} \; q_i^\top d_j .
$$

Each query token picks its single best patch. The page never has to compress "everything on it"
into one vector, which is what a single-vector embedding must do. For comparison, the
single-vector baseline here mean-pools each side first:
$s_{\text{1v}}(q, d) = \cos\big(\tfrac{1}{n}\sum_i q_i,\ \tfrac{1}{m}\sum_j d_j\big)$.

## 3.2 Training

In-batch contrastive loss over (query, page) pairs, with the other pages in the batch as negatives:

$$
\mathcal{L} = -\log \frac{\exp s(q, d^+)}{\sum_{d \in \text{batch}} \exp s(q, d)} .
$$

Queries are padded with 10 extra `<pad>` tokens (**query augmentation**). After training, these
act as learned "soft" query tokens that add matching capacity without changing the question.

## 3.3 nDCG@k with one relevant page

With a single relevant page at rank $r$ (1 = top), discounted cumulative gain at cutoff $k$
reduces to

$$
\text{nDCG@}k = \begin{cases} \dfrac{1}{\log_2(r + 1)} & r \le k \\ 0 & r > k \end{cases}
$$

averaged over queries. ViDoRe reports nDCG@5. Recall@1 is the fraction of queries whose page is
ranked first.

## 3.4 Index size

A page stores $m$ vectors of $D = 128$ dimensions. At $b$ bytes per value:

$$
\text{bytes per page} = m \cdot D \cdot b \approx 700 \times 128 \times 2 \approx 180\ \text{KB (bf16)},
$$

about 100× a single 768-d float32 vector. §6 measures how much of that can be dropped.

---

# 4) From-scratch implementation

## 4.1 A page embedding from the VLM's hidden states

The steps the model class hides:
1. Run the vision tower on the image patches.
2. Scatter the visual features into the prompt's `<|image_pad|>` slots.
3. Run the language model.
4. Project the last hidden state of **every** token to 128 dimensions and L2-normalise.

The result is checked against `model(**inputs).embeddings`.

In [ ]:
@torch.no_grad()
def embed_from_scratch(inputs: dict) -> torch.Tensor:
    """Processor outputs for one page -> (seq_len, 128) L2-normalised embeddings."""
    input_ids = inputs["input_ids"]
    # processor pads pixel_values per image; keep only this image's patches
    num_patches = int(inputs["image_grid_thw"][0].prod())
    pixel_values = inputs["pixel_values"][0, :num_patches]
    # (num_patches, 3 * 2 * 14 * 14) -> (num_patches / 4, 1536) after the 2x2 merger
    visual = model.vlm.visual(
        pixel_values, grid_thw=inputs["image_grid_thw"], return_dict=True
    ).pooler_output
    embeds = model.vlm.get_input_embeddings()(input_ids)
    image_slots = (input_ids == model.config.vlm_config.image_token_id).unsqueeze(-1)
    embeds = embeds.masked_scatter(image_slots, visual.to(embeds.dtype))
    hidden = model.vlm(inputs_embeds=embeds, attention_mask=inputs["attention_mask"])[0]
    projected = model.embedding_proj_layer(hidden)[0]
    return projected / projected.norm(dim=-1, keepdim=True)


inputs = processor(images=[pages[0]], return_tensors="pt").to(DEVICE)
ours = embed_from_scratch(inputs)
with torch.no_grad():
    reference = model(**inputs).embeddings[0]
num_image_tokens = int(
    (inputs["input_ids"] == model.config.vlm_config.image_token_id).sum()
)
grid_t, grid_h, grid_w = inputs["image_grid_thw"][0].tolist()
print(
    f"page {pages[0].size} -> patch grid {grid_h} x "
    f"{grid_w} -> {num_image_tokens} visual tokens "
    f"(2x2 merged) + {inputs['input_ids'].shape[1] - num_image_tokens} prompt tokens"
)
print(
    f"embeddings {tuple(ours.shape)} | max |ours - model| "
    f"= {(ours.float() - reference.float()).abs().max():.2e}"
)

## 4.2 Batched MaxSim

Pages have different token counts, so they are padded into one tensor with zero vectors. A zero
vector scores 0, which can never beat a real patch's cosine for a well-matched token. For safety,
padded slots are still masked to $-\infty$ before the max.

In [ ]:
def pad_embeddings(embeddings: list[torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
    """List of (len_i, dim) -> padded (n, max_len, dim) and mask (n, max_len)."""
    max_len = max(len(e) for e in embeddings)
    padded = torch.zeros(
        len(embeddings), max_len, embeddings[0].shape[1], dtype=embeddings[0].dtype
    )
    mask = torch.zeros(len(embeddings), max_len, dtype=torch.bool)
    for i, e in enumerate(embeddings):
        padded[i, : len(e)] = e
        mask[i, : len(e)] = True
    return padded, mask


def maxsim_scores(
    query_embs: list[torch.Tensor], page_embs: list[torch.Tensor]
) -> torch.Tensor:
    """(num_queries, num_pages) late-interaction scores sum_i max_j q_i . d_j."""
    q, q_mask = pad_embeddings(query_embs)
    d, d_mask = pad_embeddings(page_embs)
    q, d = q.float(), d.float()
    # (num_queries, num_pages, q_len, d_len)
    sims = torch.einsum("qid,pjd->qpij", q, d)
    sims = sims.masked_fill(~d_mask[None, :, None, :], float("-inf"))
    best = sims.max(dim=-1).values
    return (best * q_mask[:, None, :]).sum(-1)


def ndcg_and_recall(scores: torch.Tensor, k: int = 5) -> tuple[float, float]:
    """scores (n, n) with query i relevant to page i -> (nDCG@k, Recall@1)."""
    ranks = (scores > scores.diag()[:, None]).sum(1) + 1
    gains = torch.where(
        ranks <= k,
        1 / torch.log2(ranks.float() + 1),
        torch.zeros_like(ranks, dtype=torch.float),
    )
    return gains.mean().item(), (ranks == 1).float().mean().item()


toy_scores = torch.tensor([[3.0, 1.0, 0.0], [2.0, 1.0, 0.0], [0.0, 1.0, 2.0]])
print(
    "toy: query 0 rank 1, query 1 rank 2, query 2 rank "
    f"1 -> nDCG@5, R@1 = {ndcg_and_recall(toy_scores)}"
)
print(f"expected nDCG@5 = {(1 + 1 / np.log2(3) + 1) / 3:.4f}, R@1 = {2 / 3:.4f}")

## 4.3 OCR text and BM25

Textract returns a JSON list of blocks: `PAGE`, `LINE`, `WORD`, each with geometry. Joining
`LINE` blocks in reading order gives the page text an OCR pipeline would index. BM25 then scores
with term frequency, inverse document frequency and length normalisation (`rank_bm25`).

In [ ]:
def textract_text(ocr_field: str) -> str:
    """ViDoRe's stringified Textract output -> page text from LINE blocks."""
    lines = []
    for chunk in ast.literal_eval(ocr_field):
        for blocks in json.loads(chunk).values():
            lines += [b["Text"] for b in blocks if b.get("BlockType") == "LINE"]
    return "\n".join(lines)


def tokenize(text: str) -> list[str]:
    """Lowercase alphanumeric tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())


page_texts = [textract_text(row["ocr"]) for row in dataset]
print(
    "OCR words per page: median "
    f"{np.median([len(tokenize(t)) for t in page_texts]):.0f}"
)
print(page_texts[0][:400])

---

# 5) End-to-end: three retrievers on the same 100 queries

## 5.1 Indexing

In [ ]:
start = time.time()
page_embs = []
for page in pages:
    inputs = processor(images=[page], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        page_embs.append(model(**inputs).embeddings[0].cpu())
index_seconds = time.time() - start

start = time.time()
query_embs = []
for i in range(0, len(queries), 16):
    inputs = processor(text=queries[i : i + 16], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        emb = model(**inputs).embeddings.cpu()
    query_embs += [e[m.bool().cpu()] for e, m in zip(emb, inputs["attention_mask"])]
query_seconds = time.time() - start

tokens_per_page = np.array([len(e) for e in page_embs])
print(
    f"pages: {index_seconds / len(pages):.2f} s/page | queries: "
    f"{query_seconds / len(queries) * 1000:.0f} ms/query"
)
print(
    f"tokens per page: median {np.median(tokens_per_page):.0f}, "
    f"max {tokens_per_page.max()} | "
    f"index {sum(e.numel() for e in page_embs) * 2 / 1e6:.1f} "
    f"MB in bf16 | device memory {device_memory_gb():.2f} GB"
)

## 5.2 Retrieval quality

In [ ]:
colqwen_scores = maxsim_scores(query_embs, page_embs)
reference_scores = processor.score_retrieval(query_embs, page_embs)
# the library scores in bf16, ours in float32, so compare relative error
rel_diff = ((colqwen_scores - reference_scores).abs() / reference_scores.abs()).max()
print(
    "MaxSim vs processor.score_retrieval: max "
    f"relative diff {rel_diff:.2e} (bf16 rounding)"
)

mean_q = F.normalize(torch.stack([e.float().mean(0) for e in query_embs]), dim=-1)
mean_d = F.normalize(torch.stack([e.float().mean(0) for e in page_embs]), dim=-1)
single_vector_scores = mean_q @ mean_d.T

bm25 = BM25Okapi([tokenize(t) for t in page_texts])
bm25_scores = torch.tensor(np.stack([bm25.get_scores(tokenize(q)) for q in queries]))

print("retriever                        | nDCG@5 | Recall@1")
for name, scores in [
    ("OCR (Textract) + BM25", bm25_scores),
    ("ColQwen2, single vector (mean)", single_vector_scores),
    ("ColQwen2, late interaction", colqwen_scores),
]:
    ndcg, r1 = ndcg_and_recall(scores)
    print(f"{name:32s} | {ndcg:6.3f} | {r1:8.3f}")

## 5.3 Where does a query token look?

For one query, take the query token with the highest MaxSim contribution and draw its similarity
to every visual token of the relevant page as a heat map. The visual tokens form an
$(h/2) \times (w/2)$ grid in reading order.

In [ ]:
qi = 1
inputs = processor(images=[pages[qi]], return_tensors="pt")
_, grid_h, grid_w = inputs["image_grid_thw"][0].tolist()
is_image = inputs["input_ids"][0] == model.config.vlm_config.image_token_id
page_visual = page_embs[qi][is_image].float()

query_ids = processor(text=[queries[qi]], return_tensors="pt")["input_ids"][0]
query_tokens = processor.tokenizer.convert_ids_to_tokens(query_ids)
sims = query_embs[qi].float() @ page_visual.T
# skip the "Query:" prefix and the padding tokens when picking a word to show
word_positions = [
    i
    for i, t in enumerate(query_tokens)
    if t.strip("Ġ").isalpha()
    and t.strip("Ġ").lower()
    not in {"query", "the", "of", "what", "which", "is", "in", "a"}
]
best = max(word_positions, key=lambda i: sims[i].max())
heat = sims[best].reshape(grid_h // 2, grid_w // 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 9))
axes[0].imshow(pages[qi])
axes[0].set_title(queries[qi][:60], fontsize=9)
axes[1].imshow(pages[qi])
axes[1].imshow(
    heat.numpy(),
    cmap="jet",
    alpha=0.5,
    extent=(0, pages[qi].size[0], pages[qi].size[1], 0),
)
axes[1].set_title(
    f"similarity to query token '{query_tokens[best].strip('Ġ')}'", fontsize=9
)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5.4 Answering from the retrieved page

Retrieval is half of RAG. The top-ranked page image goes straight to a vision-capable LLM, with no
OCR text in between. This ViDoRe split withholds the gold answers, so check each answer against
its page by eye.

In [ ]:
def page_to_data_url(page: Image.Image, max_side: int = 1600) -> str:
    """PIL page -> base64 JPEG data URL, long side downscaled to at most max_side."""
    page = page.copy()
    page.thumbnail((max_side, max_side))
    buffer = io.BytesIO()
    page.save(buffer, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()


for qi in [0, 1, 2]:
    top_page = int(colqwen_scores[qi].argmax())
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Answer briefly from "
                        f"the page. Question: {queries[qi]}",
                    },
                    {
                        "type": "input_image",
                        "image_url": page_to_data_url(pages[top_page]),
                    },
                ],
            }
        ],
    )
    print(
        f"Q: {queries[qi]}\n   retrieved page {top_page} (correct: {top_page == qi})\n"
        f"   answer: {response.output_text.strip()}"
    )

---

# 6) Variants: shrinking the index

The multi-vector index is the price of late interaction. Two compressions:

- **Patch pooling**: average neighbouring visual tokens in $2 \times 2$ blocks of the token grid,
  re-normalise, and keep prompt tokens as they are, giving about 4× fewer vectors.
  `colpali-engine` ships a clustering variant of the same idea.
- **Binarisation**: keep only the sign of each dimension (1 bit instead of 16). The dot product of
  ±1 vectors is $D - 2\cdot\text{Hamming}$, so MaxSim runs with bit operations in production.

In [ ]:
def pool_page(
    embedding: torch.Tensor,
    input_ids: torch.Tensor,
    grid_hw: tuple[int, int],
    factor: int = 2,
) -> torch.Tensor:
    """Average visual tokens over factor x factor grid blocks; keep prompt tokens."""
    is_image = input_ids == model.config.vlm_config.image_token_id
    height, width = grid_hw[0] // 2, grid_hw[1] // 2
    visual = embedding[is_image].float().reshape(height, width, -1).permute(2, 0, 1)
    pooled = F.avg_pool2d(visual[None], factor, ceil_mode=True)[0].flatten(1).T
    return torch.cat([F.normalize(pooled, dim=-1), embedding[~is_image].float()])


pooled_embs = []
for page, emb in zip(pages, page_embs):
    inputs = processor(images=[page], return_tensors="pt")
    _, grid_h, grid_w = inputs["image_grid_thw"][0].tolist()
    pooled_embs.append(pool_page(emb, inputs["input_ids"][0], (grid_h, grid_w)))

binary_q = [torch.sign(e.float()) for e in query_embs]
binary_d = [torch.sign(e.float()) for e in page_embs]

full_bytes = sum(e.numel() for e in page_embs) * 2
print("index                      | vectors/page | size      | nDCG@5 | Recall@1")
for name, q_embs, d_embs, size in [
    ("full, bf16", query_embs, page_embs, full_bytes),
    (
        "2x2 pooled, bf16",
        query_embs,
        pooled_embs,
        sum(e.numel() for e in pooled_embs) * 2,
    ),
    ("full, 1-bit", binary_q, binary_d, full_bytes / 16),
]:
    ndcg, r1 = ndcg_and_recall(maxsim_scores(q_embs, d_embs))
    per_page = np.mean([len(e) for e in d_embs])
    print(
        f"{name:26s} | {per_page:12.0f} | "
        f"{size / 1e6:6.2f} MB | {ndcg:6.3f} | {r1:8.3f}"
    )
print(f"single 128-d vector per page for reference: {NUM_PAGES * 128 * 2 / 1e6:.3f} MB")

---

# 7) Practical notes

**Candidate generation.** Exact MaxSim against every page is $O(n \cdot m \cdot N)$. Production
systems use two stages: a cheap first pass (single-vector or pooled/binary multi-vector, or
PLAID-style centroid lookups) picks a few hundred candidates, and full MaxSim reranks them.
Vector databases with native multi-vector support (Vespa, Qdrant, Weaviate, LanceDB) do this
internally.

**Cost.** At ~700 vectors × 128 dims, bf16 is ~180 KB per page, so 1 M pages ≈ 180 GB.
Binarisation brings that to ~11 GB, and pooling plus binarisation to ~3 GB. Indexing is one VLM
forward per page, about the cost of OCR but with no layout model.

**Resolution.** Qwen2-VL keeps native aspect ratio. Very tall infographics hit the token cap
(`longest_edge`) and get downscaled, and small text becomes unreadable to the model. Splitting
tall pages into overlapping tiles is the usual fix.

**When OCR + BM25 still wins.** Exact identifiers (part numbers, legal citations, rare names) and
long text-only documents favour lexical search. Hybrid retrieval, fusing BM25 and late-interaction
ranks, is a robust default.

**Generation.** Send the retrieved page image, not its OCR, to a vision-capable LLM. Send 1–3 pages,
since each costs ~1k image tokens.

---

# 8) Conclusion

| Situation | Retriever | Why |
|---|---|---|
| Visually rich pages: charts, infographics, slides | ColQwen2 / ColPali late interaction | Patches keep the visual evidence; no OCR pipeline |
| Very large corpus | Pooled + binarised multi-vector, rerank with full MaxSim | 10–50× smaller index for a small accuracy cost |
| Exact strings, text-heavy contracts | OCR + BM25 (or hybrid) | Lexical match on identifiers |
| Simplest possible deployment | Single-vector VLM embedding (e.g. jina-v4 single mode) | One vector per page in any vector DB, lower accuracy |